# TRAIN EyePACS v2 — ViT-Base, all seeds, stabilized recipe

Trains only the two ViT-Base models, for replicates 0, 1 and 2 (six models), using the
stabilized ViT recipe established earlier in this project: lr 3e-5, 5-epoch linear warmup
then cosine, layer-wise LR decay 0.65 per block, weight decay 0.05, gradient clipping 1.0.

Why a separate notebook: `TRAIN_eyepacs_v2_ALLSEEDS.ipynb` failed on ViT-Base because its
block finder looked only for `nn.ModuleList` (HuggingFace ViT) and timm's ViT stores blocks
in an `nn.Sequential`. Fixed here, and Cell 1 now preflights the timm ViT path.

**Safe to run alongside the other training run**: it only ever touches ViT-Base files, and
writes its heartbeat to a separate `TRAINING_STATUS_vit.json` so the two do not collide.
Replicate 0 keeps canonical names; replicates 1 and 2 carry `_seed1` / `_seed2`.

About 10 minutes per model on an A100. Run cells 1-5.

In [1]:
# Cell 1: auth, deps, Drive helpers, fixed block finder, timm ViT PREFLIGHT
!pip -q install timm
import io, os, json, time, zipfile, glob, datetime, random, math
import numpy as np, pandas as pd, torch
import torch.nn as nn
import timm
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
svc = gbuild('drive', 'v3')

F_SPLITS      = '13x_CS7N_UasGRSuNYMyD5To2RTYjL-EP'
F_CHECKPOINTS = '1DAAXJ2fjk2zAzx7PpGaWsk4BXudeA5Le'
F_PREDICTIONS = '1nCloJ5o9veZb2xJL_2EjdVHQXN31jy4w'
F_AUDIT       = '1zuvfcxC3umHAq_uf7Vx9omMEgUEuQhTt'
SPLIT_IDS = {'train': '1FU30c6vRYMLqX8h9z2F58iEqow1Zx8NQ',
             'val':   '1Z9tFlAGe0j4RVRgG9-6LnhTrukvFy87y',
             'test':  '1mSlAqeNtFJ0Ap6BMenrrQxHaWep-9nEj'}
IMAGE_ZIPS = {'batch1': '11WspNNHS73OadbW-sqy7BSzcrsBkX1d6',
              'batch2': '1zG7rHjm8Ty9LHG4LqMzgnOzrT_90VO9H',
              'batch3': '1L88vUNSxWjdGPBquhknZhafcnXXRd3ES',
              'batch4': '1-FrpGB74qEGzDvXlA9FhmB2OeeZkKPAN',
              'batch5': '1mEIrSGVExZ6zmxnRLxyFab5yB1kH5YBT'}
NUM_WORKERS = min(8, os.cpu_count())
STATUS_FILE = 'TRAINING_STATUS_vit.json'

def dl(fid, path):
    with io.FileIO(path, 'wb') as fh:
        d = MediaIoBaseDownload(fh, svc.files().get_media(fileId=fid), chunksize=50*1024*1024); done = False
        while not done: _, done = d.next_chunk()

def find_id(name, folder):
    r = svc.files().list(q=f"'{folder}' in parents and name='{name}' and trashed=false",
                         fields='files(id)').execute().get('files', [])
    return r[0]['id'] if r else None

def up(path, folder, mime='application/octet-stream', overwrite=False):
    media = MediaFileUpload(path, mimetype=mime, resumable=True); name = os.path.basename(path)
    if overwrite:
        fid = find_id(name, folder)
        if fid: svc.files().update(fileId=fid, media_body=media).execute(); return
    svc.files().create(body={'name': name, 'parents': [folder]}, media_body=media, fields='id').execute()
    print('  uploaded', name)

def existing(folder):
    names, tok = set(), None
    while True:
        r = svc.files().list(q=f"'{folder}' in parents and trashed=false",
                             fields='files(name),nextPageToken', pageToken=tok, pageSize=1000).execute()
        names |= {f['name'] for f in r.get('files', [])}; tok = r.get('nextPageToken')
        if not tok: break
    return names

STATUS = {'started': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'run': 'ViT-Base all seeds', 'events': []}
def heartbeat(**kw):
    STATUS.update(kw); STATUS['updated'] = datetime.datetime.now(datetime.timezone.utc).isoformat()
    with open(STATUS_FILE, 'w') as f: json.dump(STATUS, f, indent=2)
    try: up(STATUS_FILE, F_AUDIT, 'application/json', overwrite=True)
    except Exception as e: print('  (heartbeat failed, continuing):', e)

def find_blocks(m):
    """Transformer block container: nn.ModuleList (HuggingFace ViT) or nn.Sequential (timm ViT)."""
    best = None
    for _, mod in m.named_modules():
        if isinstance(mod, (nn.ModuleList, nn.Sequential)) and len(mod) >= 12 and (best is None or len(mod) > len(best)):
            best = mod
    if best is None: raise RuntimeError('could not locate transformer blocks')
    return best

# ---- PREFLIGHT: the exact timm ViT path used below ----
_v = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=1)
_b = find_blocks(_v)
_names = [n for n, _ in _v.named_parameters()]
assert len(_b) == 12, f'expected 12 ViT-Base blocks, found {len(_b)}'
assert any(n.startswith('head') for n in _names), 'no head parameters found'
assert any(n.startswith(('cls_token', 'pos_embed', 'patch_embed')) for n in _names), 'no embedding parameters found'
print(f'GPU: {torch.cuda.get_device_name(0)} | timm ViT preflight passed: {len(_b)} blocks, '
      f'{sum(p.numel() for p in _v.parameters())/1e6:.1f}M params'); del _v, _b

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

GPU: Tesla T4 | timm ViT preflight passed: 12 blocks, 85.8M params


In [2]:
# Cell 2: ViT-Base roster (stabilized recipe) and replicates
VIT_STABLE = {'lr': 3e-5, 'wd': 0.05, 'clip': 1.0, 'lldecay': 0.65, 'warmup': 5, 'recipe': 'vit_stable'}
MODELS = [
  {'name':'vit_base_binary','arch':'vit_base_patch16_224','task':'binary','nc':1,'epochs':15,'bs':32, **VIT_STABLE},
  {'name':'vit_base_5class','arch':'vit_base_patch16_224','task':'5class','nc':5,'epochs':15,'bs':32, **VIT_STABLE},
]
# roster index must match the full 11-model roster so seeds equal the canonical scheme
FULL_ROSTER = ['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class',
               'resnet50_binary','resnet50_5class','vit_base_binary','vit_base_5class',
               'retfound_binary','retfound_5class','retfound_adversarial']
REPLICATES = [0, 1, 2]
BASE_SEED = 20260920
IMG_SIZE = 224
def tag(k): return '' if k == 0 else f'_seed{k}'
print('ViT recipe:', VIT_STABLE)

ViT recipe: {'lr': 3e-05, 'wd': 0.05, 'clip': 1.0, 'lldecay': 0.65, 'warmup': 5, 'recipe': 'vit_stable'}


In [3]:
# Cell 3: fetch splits and images, verify coverage
os.makedirs('/content/img', exist_ok=True)
splits = {}
for s, fid in SPLIT_IDS.items():
    p = f'split_{s}.csv'
    if not os.path.exists(p): dl(fid, p)
    splits[s] = pd.read_csv(p)
heartbeat(stage='downloading images')
if not glob.glob('/content/img/**/*.jp*g', recursive=True):
    for name, fid in IMAGE_ZIPS.items():
        z = f'/content/{name}.zip'
        if not os.path.exists(z): print('downloading', name); dl(fid, z)
        with zipfile.ZipFile(z) as zf: zf.extractall('/content/img')
        os.remove(z); heartbeat(stage=f'extracted {name}')
paths = glob.glob('/content/img/**/*.jp*g', recursive=True) + glob.glob('/content/img/**/*.png', recursive=True)
INDEX = {os.path.splitext(os.path.basename(p))[0]: p for p in paths}
need = set().union(*[set(d['image_id'].astype(str)) for d in splits.values()])
missing = sorted(need - set(INDEX))
print('images on disk:', len(INDEX), '| missing:', len(missing))
if missing: raise SystemExit('Images missing. Stop and report.')

  uploaded TRAINING_STATUS_vit.json
downloading batch1
downloading batch2
downloading batch3
downloading batch4
downloading batch5
images on disk: 19332 | missing: 0


In [4]:
# Cell 4: dataset, ViT LLRD optimizer, warmup+cosine scheduler, train, predict
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import roc_auc_score
from PIL import Image
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]
TF_TRAIN = transforms.Compose([transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.ColorJitter(0.2,0.2,0.1), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
TF_EVAL = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class DR(Dataset):
    def __init__(self, df, tf): self.df = df.reset_index(drop=True); self.tf = tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return self.tf(Image.open(INDEX[str(r['image_id'])]).convert('RGB')), int(r['level']), int(r['binary_label']), i

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def make_optimizer(model, cfg):
    lr, d = cfg['lr'], cfg['lldecay']
    blocks = find_blocks(model); n = len(blocks)
    named = list(model.named_parameters())
    bid = {id(p) for b in blocks for p in b.parameters()}
    embed = [p for nm, p in named if id(p) not in bid and nm.startswith(('cls_token','pos_embed','patch_embed','reg_token'))]
    head = [p for nm, p in named if id(p) not in bid and nm.startswith('head')]
    used = bid | {id(p) for p in embed} | {id(p) for p in head}
    rest = [p for nm, p in named if id(p) not in used]
    groups = [{'params': embed, 'lr': lr * d ** n}]
    groups += [{'params': list(b.parameters()), 'lr': lr * d ** (n - 1 - i)} for i, b in enumerate(blocks)]
    groups += [{'params': rest, 'lr': lr}, {'params': head, 'lr': lr}]
    groups = [g for g in groups if g['params']]
    total = sum(p.numel() for g in groups for p in g['params'])
    assert total == sum(p.numel() for p in model.parameters()), 'LLRD groups do not cover every parameter'
    print(f'  ViT LLRD {d}: {len(groups)} groups, {lr*d**n:.2e} (embed) -> {lr:.2e} (head)')
    return torch.optim.AdamW(groups, lr=lr, weight_decay=cfg['wd'])

def make_scheduler(opt, cfg):
    E, W = cfg['epochs'], cfg['warmup']
    f = lambda ep: (ep + 1) / W if ep < W else 0.5 * (1 + math.cos(math.pi * (ep - W) / max(1, E - W)))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

@torch.no_grad()
def predict(model, df, dev, bs=64):
    model.eval()
    dl_ = DataLoader(DR(df, TF_EVAL), batch_size=bs, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    probs = np.zeros(len(df), dtype=np.float64)
    for x, _, _, idx in dl_:
        out = model(x.to(dev, non_blocking=True)).float()
        p = torch.sigmoid(out[:, 0]) if out.shape[1] == 1 else torch.softmax(out, 1)[:, 2:].sum(1)
        probs[idx.numpy()] = p.double().cpu().numpy()
    return probs

def train_one(cfg, dev, seed, label):
    seed_everything(seed)
    tr, va = splits['train'], splits['val']
    model = timm.create_model(cfg['arch'], pretrained=True, num_classes=cfg['nc']).to(dev)
    g = torch.Generator(); g.manual_seed(seed)
    dl_ = DataLoader(DR(tr, TF_TRAIN), batch_size=cfg['bs'], shuffle=True, num_workers=NUM_WORKERS,
                     drop_last=True, generator=g, pin_memory=True, persistent_workers=True, prefetch_factor=4)
    opt = make_optimizer(model, cfg); sch = make_scheduler(opt, cfg)
    lossf = nn.BCEWithLogitsLoss() if cfg['nc'] == 1 else nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda')
    best_auc, best_state, best_ep = -1.0, None, -1
    for ep in range(cfg['epochs']):
        model.train(); tot = 0.0; t0 = time.time()
        for x, lvl, lab, idx in dl_:
            x = x.to(dev, non_blocking=True)
            y = (lab.float() if cfg['nc'] == 1 else lvl.long()).to(dev, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                out = model(x)
                loss = lossf(out[:, 0], y) if cfg['nc'] == 1 else lossf(out, y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['clip'])
            scaler.step(opt); scaler.update(); tot += float(loss.item())
        sch.step()
        vp = predict(model, va, dev); vauc = float(roc_auc_score(va['binary_label'].values, vp))
        star = ''
        if vauc > best_auc:
            best_auc, best_ep = vauc, ep + 1
            best_state = {k_: v.detach().cpu().clone() for k_, v in model.state_dict().items()}; star = '  <- best'
        print(f"  epoch {ep+1}/{cfg['epochs']}  loss {tot/max(1,len(dl_)):.4f}  val_auc {vauc:.4f}  "
              f"lr {opt.param_groups[-1]['lr']:.2e}  ({time.time()-t0:.0f}s){star}")
        heartbeat(stage='training', current=label, epoch=f"{ep+1}/{cfg['epochs']}",
                  last_val_auc=round(vauc, 4), best_val_auc=round(best_auc, 4), best_epoch=best_ep)
    model.load_state_dict(best_state)
    return model, best_auc, best_ep

In [5]:
# Cell 5: run all ViT-Base (replicate, model) pairs not yet checkpointed
dev = 'cuda'
have = existing(F_CHECKPOINTS)
todo = [(k, m) for k in REPLICATES for m in MODELS if f"eyepacs_{m['name']}{tag(k)}_v2.pt" not in have]
heartbeat(stage='starting', queued=[m['name'] + (tag(k) or '_seed0') for k, m in todo])
print(f'to train: {len(todo)} of {len(REPLICATES) * len(MODELS)}')

for i, (k, cfg) in enumerate(todo, 1):
    seed = BASE_SEED + 1000 * k + FULL_ROSTER.index(cfg['name'])
    label = cfg['name'] + (tag(k) or '_seed0')
    print(f'\n=== [{i}/{len(todo)}] {label} (seed {seed}) ===')
    t0 = time.time()
    try:
        model, best_auc, best_ep = train_one(cfg, dev, seed, label)
    except Exception as e:
        print('  FAILED:', e); STATUS['events'].append({'model': label, 'status': 'failed', 'error': str(e)})
        heartbeat(stage='model failed', current=label); continue
    ck = f"eyepacs_{cfg['name']}{tag(k)}_v2.pt"
    torch.save({'state_dict': model.state_dict(), 'config': cfg, 'seed': seed, 'replicate': k,
                'best_val_auc': best_auc, 'best_epoch': best_ep}, ck)
    up(ck, F_CHECKPOINTS)
    for s in ['train', 'val', 'test']:
        d = splits[s]; pr = predict(model, d, dev)
        fn = f"eyepacs_predictions_{s}_{cfg['name']}{tag(k)}_v2.csv"
        pd.DataFrame({'image_id': d['image_id'].astype(str), 'patient_id': d['patient_id'].astype(str),
                      'eye': d['eye'], 'dataset': 'eyepacs', 'split': s,
                      'true_grade': d['level'].astype(int), 'true_label': d['binary_label'].astype(int),
                      'predicted_probability': pr, 'predicted_class': (pr >= 0.5).astype(int),
                      'model_name': cfg['name'], 'replicate': k}).to_csv(fn, index=False, float_format='%.17g')
        up(fn, F_PREDICTIONS, 'text/csv')
    mins = round((time.time() - t0) / 60, 1)
    STATUS['events'].append({'model': label, 'status': 'ok', 'minutes': mins,
                             'best_val_auc': round(best_auc, 4), 'best_epoch': best_ep, 'seed': seed})
    heartbeat(stage='model complete', current=label)
    print(f'  done in {mins} min (best val_auc {best_auc:.4f} @ epoch {best_ep})')
    del model; torch.cuda.empty_cache()

done = existing(F_CHECKPOINTS)
left = [m['name'] + (tag(k) or '_seed0') for k in REPLICATES for m in MODELS if f"eyepacs_{m['name']}{tag(k)}_v2.pt" not in done]
heartbeat(stage='session finished', remaining=left)
print(f'\nremaining ViT models: {len(left)}', left)

to train: 6 of 6

=== [1/6] vit_base_binary_seed0 (seed 20260926) ===
  ViT LLRD 0.65: 15 groups, 1.71e-07 (embed) -> 3.00e-05 (head)
  epoch 1/15  loss 0.4336  val_auc 0.7850  lr 1.20e-05  (304s)  <- best
  epoch 2/15  loss 0.3799  val_auc 0.8281  lr 1.80e-05  (274s)  <- best
  epoch 3/15  loss 0.3563  val_auc 0.8486  lr 2.40e-05  (265s)  <- best
  epoch 4/15  loss 0.3289  val_auc 0.8559  lr 3.00e-05  (263s)  <- best
  epoch 5/15  loss 0.3091  val_auc 0.8606  lr 3.00e-05  (264s)  <- best
  epoch 6/15  loss 0.2917  val_auc 0.8612  lr 2.93e-05  (263s)  <- best
  epoch 7/15  loss 0.2739  val_auc 0.8583  lr 2.71e-05  (263s)
  epoch 8/15  loss 0.2525  val_auc 0.8571  lr 2.38e-05  (261s)
  epoch 9/15  loss 0.2339  val_auc 0.8683  lr 1.96e-05  (260s)  <- best
  epoch 10/15  loss 0.2072  val_auc 0.8509  lr 1.50e-05  (262s)
  epoch 11/15  loss 0.1899  val_auc 0.8520  lr 1.04e-05  (261s)


  epoch 12/15  loss 0.1677  val_auc 0.8523  lr 6.18e-06  (256s)
  epoch 13/15  loss 0.1449  val_auc 0.8498  lr 2.86e-06  (259s)
  epoch 14/15  loss 0.1321  val_auc 0.8520  lr 7.34e-07  (255s)
  epoch 15/15  loss 0.1280  val_auc 0.8509  lr 0.00e+00  (253s)
  uploaded eyepacs_vit_base_binary_v2.pt
  uploaded eyepacs_predictions_train_vit_base_binary_v2.csv
  uploaded eyepacs_predictions_val_vit_base_binary_v2.csv
  uploaded eyepacs_predictions_test_vit_base_binary_v2.csv
  done in 71.4 min (best val_auc 0.8683 @ epoch 9)

=== [2/6] vit_base_5class_seed0 (seed 20260927) ===
  ViT LLRD 0.65: 15 groups, 1.71e-07 (embed) -> 3.00e-05 (head)
  epoch 1/15  loss 0.8159  val_auc 0.7814  lr 1.20e-05  (252s)  <- best
  epoch 2/15  loss 0.7220  val_auc 0.8050  lr 1.80e-05  (254s)  <- best
  epoch 3/15  loss 0.6853  val_auc 0.8416  lr 2.40e-05  (254s)  <- best
  epoch 4/15  loss 0.6536  val_auc 0.8551  lr 3.00e-05  (251s)  <- best
  epoch 5/15  loss 0.6226  val_auc 0.8515  lr 3.00e-05  (251s)
  epoch

  epoch 10/15  loss 0.4320  val_auc 0.8564  lr 1.50e-05  (254s)
  epoch 11/15  loss 0.3875  val_auc 0.8551  lr 1.04e-05  (255s)
  epoch 12/15  loss 0.3473  val_auc 0.8387  lr 6.18e-06  (255s)
  epoch 13/15  loss 0.3209  val_auc 0.8370  lr 2.86e-06  (270s)
  epoch 14/15  loss 0.2875  val_auc 0.8454  lr 7.34e-07  (259s)
  epoch 15/15  loss 0.2675  val_auc 0.8451  lr 0.00e+00  (261s)
  uploaded eyepacs_vit_base_5class_v2.pt
  uploaded eyepacs_predictions_train_vit_base_5class_v2.csv
  uploaded eyepacs_predictions_val_vit_base_5class_v2.csv
  uploaded eyepacs_predictions_test_vit_base_5class_v2.csv
  done in 69.2 min (best val_auc 0.8632 @ epoch 6)

=== [3/6] vit_base_binary_seed1 (seed 20261926) ===
  ViT LLRD 0.65: 15 groups, 1.71e-07 (embed) -> 3.00e-05 (head)
  epoch 1/15  loss 0.4374  val_auc 0.7818  lr 1.20e-05  (271s)  <- best
  epoch 2/15  loss 0.3759  val_auc 0.8178  lr 1.80e-05  (273s)  <- best
  epoch 3/15  loss 0.3523  val_auc 0.8361  lr 2.40e-05  (261s)  <- best
  epoch 4/15  

  epoch 7/15  loss 0.2741  val_auc 0.8652  lr 2.71e-05  (266s)
  epoch 8/15  loss 0.2532  val_auc 0.8630  lr 2.38e-05  (265s)
  epoch 9/15  loss 0.2375  val_auc 0.8639  lr 1.96e-05  (263s)
  epoch 10/15  loss 0.2159  val_auc 0.8589  lr 1.50e-05  (262s)
  epoch 11/15  loss 0.1957  val_auc 0.8514  lr 1.04e-05  (258s)
  epoch 12/15  loss 0.1729  val_auc 0.8552  lr 6.18e-06  (257s)
  epoch 13/15  loss 0.1560  val_auc 0.8517  lr 2.86e-06  (253s)
  epoch 14/15  loss 0.1403  val_auc 0.8510  lr 7.34e-07  (252s)
  epoch 15/15  loss 0.1330  val_auc 0.8486  lr 0.00e+00  (254s)
  uploaded eyepacs_vit_base_binary_seed1_v2.pt
  uploaded eyepacs_predictions_train_vit_base_binary_seed1_v2.csv
  uploaded eyepacs_predictions_val_vit_base_binary_seed1_v2.csv
  uploaded eyepacs_predictions_test_vit_base_binary_seed1_v2.csv
  done in 71.1 min (best val_auc 0.8710 @ epoch 6)

=== [4/6] vit_base_5class_seed1 (seed 20261927) ===
  ViT LLRD 0.65: 15 groups, 1.71e-07 (embed) -> 3.00e-05 (head)
  epoch 1/15  los

  epoch 4/15  loss 0.6505  val_auc 0.8493  lr 3.00e-05  (260s)  <- best
  epoch 5/15  loss 0.6219  val_auc 0.8645  lr 3.00e-05  (259s)  <- best
  epoch 6/15  loss 0.5795  val_auc 0.8714  lr 2.93e-05  (254s)  <- best
  epoch 7/15  loss 0.5558  val_auc 0.8543  lr 2.71e-05  (252s)
  epoch 8/15  loss 0.5231  val_auc 0.8577  lr 2.38e-05  (249s)
  epoch 9/15  loss 0.4805  val_auc 0.8620  lr 1.96e-05  (254s)
  epoch 10/15  loss 0.4333  val_auc 0.8551  lr 1.50e-05  (257s)
  epoch 11/15  loss 0.3874  val_auc 0.8531  lr 1.04e-05  (255s)
  epoch 12/15  loss 0.3503  val_auc 0.8481  lr 6.18e-06  (254s)
  epoch 13/15  loss 0.3097  val_auc 0.8436  lr 2.86e-06  (253s)
  epoch 14/15  loss 0.2889  val_auc 0.8420  lr 7.34e-07  (251s)
  epoch 15/15  loss 0.2762  val_auc 0.8448  lr 0.00e+00  (252s)
  uploaded eyepacs_vit_base_5class_seed1_v2.pt
  uploaded eyepacs_predictions_train_vit_base_5class_seed1_v2.csv
  uploaded eyepacs_predictions_val_vit_base_5class_seed1_v2.csv
  uploaded eyepacs_predictions_tes

  epoch 2/15  loss 0.3738  val_auc 0.8245  lr 1.80e-05  (255s)  <- best
  epoch 3/15  loss 0.3478  val_auc 0.8259  lr 2.40e-05  (254s)  <- best
  epoch 4/15  loss 0.3350  val_auc 0.8574  lr 3.00e-05  (254s)  <- best
  epoch 5/15  loss 0.3079  val_auc 0.8394  lr 3.00e-05  (254s)
  epoch 6/15  loss 0.2900  val_auc 0.8573  lr 2.93e-05  (254s)
  epoch 7/15  loss 0.2663  val_auc 0.8585  lr 2.71e-05  (252s)  <- best
  epoch 8/15  loss 0.2512  val_auc 0.8653  lr 2.38e-05  (251s)  <- best
  epoch 9/15  loss 0.2297  val_auc 0.8506  lr 1.96e-05  (248s)
  epoch 10/15  loss 0.2056  val_auc 0.8553  lr 1.50e-05  (251s)
  epoch 11/15  loss 0.1799  val_auc 0.8425  lr 1.04e-05  (255s)
  epoch 12/15  loss 0.1630  val_auc 0.8458  lr 6.18e-06  (254s)
  epoch 13/15  loss 0.1428  val_auc 0.8477  lr 2.86e-06  (254s)
  epoch 14/15  loss 0.1309  val_auc 0.8464  lr 7.34e-07  (255s)
  epoch 15/15  loss 0.1234  val_auc 0.8481  lr 0.00e+00  (256s)
  uploaded eyepacs_vit_base_binary_seed2_v2.pt


  uploaded eyepacs_predictions_train_vit_base_binary_seed2_v2.csv
  uploaded eyepacs_predictions_val_vit_base_binary_seed2_v2.csv
  uploaded eyepacs_predictions_test_vit_base_binary_seed2_v2.csv
  done in 68.7 min (best val_auc 0.8653 @ epoch 8)

=== [6/6] vit_base_5class_seed2 (seed 20262927) ===
  ViT LLRD 0.65: 15 groups, 1.71e-07 (embed) -> 3.00e-05 (head)
  epoch 1/15  loss 0.8282  val_auc 0.7826  lr 1.20e-05  (260s)  <- best
  epoch 2/15  loss 0.7230  val_auc 0.8154  lr 1.80e-05  (256s)  <- best
  epoch 3/15  loss 0.6786  val_auc 0.8369  lr 2.40e-05  (257s)  <- best
  epoch 4/15  loss 0.6580  val_auc 0.8457  lr 3.00e-05  (258s)  <- best
  epoch 5/15  loss 0.6249  val_auc 0.8562  lr 3.00e-05  (258s)  <- best
  epoch 6/15  loss 0.5877  val_auc 0.8663  lr 2.93e-05  (256s)  <- best
  epoch 7/15  loss 0.5585  val_auc 0.8628  lr 2.71e-05  (255s)
  epoch 8/15  loss 0.5209  val_auc 0.8632  lr 2.38e-05  (254s)
  epoch 9/15  loss 0.4764  val_auc 0.8651  lr 1.96e-05  (255s)
  epoch 10/15  l

  epoch 13/15  loss 0.3201  val_auc 0.8536  lr 2.86e-06  (265s)
  epoch 14/15  loss 0.2961  val_auc 0.8547  lr 7.34e-07  (264s)
  epoch 15/15  loss 0.2813  val_auc 0.8543  lr 0.00e+00  (260s)
  uploaded eyepacs_vit_base_5class_seed2_v2.pt
  uploaded eyepacs_predictions_train_vit_base_5class_seed2_v2.csv
  uploaded eyepacs_predictions_val_vit_base_5class_seed2_v2.csv
  uploaded eyepacs_predictions_test_vit_base_5class_seed2_v2.csv
  done in 70.0 min (best val_auc 0.8663 @ epoch 6)

remaining ViT models: 0 []
